In [1]:
# %%
import json
import re
import pathlib
import multiprocessing as mp
import xarray as xr
import tqdm

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from functools import partial
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

In [2]:
# %%
src_path = pathlib.Path('/gpfs/work2/0/prjs1027/swn-runs/')
boundary_path = pathlib.Path('/gpfs/work2/0/prjs1027/guus/data/BoundaryConditions/bnd_storms.nc')
bathymetry_path = pathlib.Path('/gpfs/work2/0/prjs1027/swn-runs/02-2022/geometry/swan-ns-j22_6-v1a_adjust.bot')

write_dir = pathlib.Path('/gpfs/work2/0/prjs1027/tfrecords/run3')
write_dir.mkdir(exist_ok=True, parents=True)
filtered_paths = [path for path in src_path.glob('ens*') if re.search(r'\d+$', str(path))]
filtered_paths = [path for path in src_path.glob('0*') if path.stem[-4:] != '2022']

In [3]:
# %%
boundary_ds = xr.open_dataset(boundary_path)
dummy_ds = xr.open_dataset(next(filtered_paths[0].glob('computations/ens0/*/output/*.nc')))

In [4]:
# %%
output_variables = ["hs", "theta0", "tmm10", "tps", "hswe"]
input_variables = ["xwnd", "ywnd", "wl", "xcur", "ycur"]
boundary_variables = ["hs", "hswe", "tmm10", "theta0"]
conversion_dict = {"hs": "Hm0", "hswe": "HE10", "tmm10": "Tmm10", "theta0": "meanD"}

In [5]:
def load_ascii_bathy(input_grid_file_path, lon_start, lat_start, lon_step, lat_step):
    """
    Loads bathymetry and grid from ascii file (GEBCO)
 
    Args:
        input_grid_file_path (str): path to grid file
        lon_start: start longitude
        lat_start:
    Returns:
        lon, lat, bottom_data (np.array): 2D arrays with lon and lat values
        and bottom data
 
    Example usage:
        lon_start, lat_start, lon_step, lat_step = -3.000000, 51.500000, 0.500000, 0.500000
        lon_bathy, lat_bathy, bottom_data = input_loader.load_bathy(extra_path + "depth.GRD", lon_start, lat_start, lon_step, lat_step)
 
    """
    # Read the bottom grid data from the ASCII file
    bottom_data = np.loadtxt(input_grid_file_path)
 
    # Transpose the data to match the provided organization (latitude columns, longitude rows)
    bottom_data = bottom_data.T
 
    # Replace -999 with NaN
    bottom_data[bottom_data == -999] = np.nan
 
    lons = np.arange(lon_start, lon_start + len(bottom_data[0]) * lon_step, lon_step)
    lats = np.arange(lat_start, lat_start + len(bottom_data) * lat_step, lat_step)
 
    # Create a meshgrid from the coordinates
    lon, lat = np.meshgrid(lons, lats)
 
    return lon, lat, bottom_data

bathymetry = load_ascii_bathy(bathymetry_path, -12, 48, 0.05, 1/30)[2]

dummy_ds['wl'] = dummy_ds['ssh'] + bathymetry

In [6]:
def correct_nan_theta0(ds):
    for i in range(ds.time.shape[0]):
        hs = ds['hs'].values[i, ...]
        theta0 = ds['theta0'].values[i, ...]
        mask_hs = ma.masked_invalid(hs).mask
        mask_theta0 = ma.masked_invalid(theta0).mask
        
        invalid_ns_mask = np.logical_xor(mask_hs, mask_theta0)
        if invalid_ns_mask.sum() > 0:
            # Extract valid data points
            x = np.arange(theta0.shape[1])
            y = np.arange(theta0.shape[0])
            
            xx, yy = np.meshgrid(x, y)
            x1 = xx[~invalid_ns_mask]
            y1 = yy[~invalid_ns_mask]
            
            new_data = theta0[~invalid_ns_mask]
                    
            # Create interpolator for valid data
            interpolated = griddata((x1, y1), new_data.ravel(), (xx, yy), method='nearest', fill_value=None)
            ds['theta0'].values[i, ...] = interpolated
    return ds

def correct_outlier_cur(ds):
    for i in range(ds.time.shape[0]):
        xcur = ds['xcur'].values[i, ...]
        ycur = ds['ycur'].values[i, ...]
        
        outliers_xcur = np.abs(xcur) > 3
        outliers_ycur = np.abs(ycur) > 3

        xcur[outliers_xcur] = np.nanmean(xcur[~outliers_xcur])
        ycur[outliers_ycur] = np.nanmean(ycur[~outliers_ycur])
        
        ds['xcur'].values[i, ...] = xcur
        ds['ycur'].values[i, ...] = ycur

    return ds
        

In [7]:
# %%
def compute_min_max(path):
    all_datasets = []
    min_values = {}
    max_values = {}
    
    ens_paths_50 = [ens_path for ens_path in path.glob('computations/ens*') if re.search(r'\d+$', str(ens_path))]
    ens_paths = []
    for path in ens_paths_50:
        ens = re.search(r'\d+$', str(path)).group(0)
        if int(ens) in range(30):
            # ens_paths.append(path)
            
            for var in dummy_ds.variables:
                min_values[var] = []
                max_values[var] = []
            
            for _ in tqdm.tqdm(list(path.glob('*'))):
                try:
                    ds_path = next(_.glob('output/*.nc'))
                    ds = xr.open_dataset(ds_path)

                    ds['wl'] = ds['ssh'] + bathymetry
                    ds = correct_outlier_cur(ds)
                    for var in ds.variables:
                        min_values[var].append(ds[var].min().values)
                        max_values[var].append(ds[var].max().values)
                except:
                    continue
        
            min_max_values = {}
            for var in dummy_ds.variables:
                min_value = np.nanmin(min_values[var])
                max_value = np.nanmax(max_values[var])
                min_max_values[var] = np.array([min_value, max_value])
            return min_max_values

In [8]:
# %%
min_max_values = {}
for paths in filtered_paths:
    min_max_values[paths.stem] = compute_min_max(paths)

  0%|          | 0/27 [00:00<?, ?it/s]/scratch-local/gvhemert1.11159962/ipykernel_1864581/3526046899.py:34: RuntimeWarning: Mean of empty slice
  ycur[outliers_ycur] = np.nanmean(ycur[~outliers_ycur])
 41%|████      | 11/27 [00:05<00:07,  2.05it/s]/scratch-local/gvhemert1.11159962/ipykernel_1864581/3526046899.py:34: RuntimeWarning: Mean of empty slice
  ycur[outliers_ycur] = np.nanmean(ycur[~outliers_ycur])
 67%|██████▋   | 18/27 [00:08<00:04,  2.13it/s]/scratch-local/gvhemert1.11159962/ipykernel_1864581/3526046899.py:34: RuntimeWarning: Mean of empty slice
  ycur[outliers_ycur] = np.nanmean(ycur[~outliers_ycur])
100%|██████████| 30/30 [00:18<00:00,  1.61it/s]
/scratch-local/gvhemert1.11159962/ipykernel_1864581/1196921629.py:33: RuntimeWarning: All-NaN axis encountered
  min_value = np.nanmin(min_values[var])
/scratch-local/gvhemert1.11159962/ipykernel_1864581/1196921629.py:34: RuntimeWarning: All-NaN axis encountered
  max_value = np.nanmax(max_values[var])
100%|██████████| 6/6 [00:03

In [9]:
# %%
min_max_values_detailed = {}
for ens in min_max_values.keys():
    for var in min_max_values[ens].keys():
        try:
            min_max_values_detailed[var] = np.concatenate([min_max_values_detailed[var], min_max_values[ens][var]])
        except:
            min_max_values_detailed[var] = min_max_values[ens][var]

In [10]:
# %%
for var in min_max_values_detailed.keys():
    min_value = np.nanmin(min_max_values_detailed[var])
    max_value = np.nanmax(min_max_values_detailed[var])
    min_max_values_detailed[var] = {
        "minimum": min_value,
        "maximum": max_value
    }

In [11]:
min_max_values_detailed

{'time': {'minimum': numpy.datetime64('2013-12-01T00:00:00.000000000'),
  'maximum': numpy.datetime64('2023-12-31T00:00:00.000000000')},
 'longitude': {'minimum': -12.0, 'maximum': 9.0},
 'latitude': {'minimum': 48.0, 'maximum': 64.0},
 'hs': {'minimum': 0.0, 'maximum': 13.415173530578613},
 'hswe': {'minimum': 0.0, 'maximum': 12.44392204284668},
 'tmm10': {'minimum': 0.0, 'maximum': 50.0},
 'tps': {'minimum': 1.6388435363769531, 'maximum': 33.33308029174805},
 'theta0': {'minimum': 0.0, 'maximum': 360.0},
 'fdir': {'minimum': 0.0, 'maximum': 360.0},
 'spread': {'minimum': 0.0, 'maximum': 81.02664184570312},
 'ssh': {'minimum': -6.4157538414001465, 'maximum': 7.284157752990723},
 'xcur': {'minimum': -2.999359130859375, 'maximum': 2.996917724609375},
 'ycur': {'minimum': -2.999359130859375, 'maximum': 2.999969482421875},
 'xwnd': {'minimum': -28.125858306884766, 'maximum': 33.298744201660156},
 'ywnd': {'minimum': -31.77281951904297, 'maximum': 32.187870025634766},
 'wl': {'minimum': -7

In [12]:
# %%
df = pd.DataFrame(min_max_values_detailed)

In [13]:
# %%
ds_min_max = xr.Dataset.from_dataframe(df)
ds_min_max_dir = write_dir / "min-max-values"
ds_min_max_dir.mkdir(exist_ok=True, parents=True)
ds_min_max.to_netcdf(ds_min_max_dir / "min-max-values-storms.nc")

In [15]:
ds_min_max

<xarray.Dataset>
Dimensions:    (index: 2)
Coordinates:
  * index      (index) object 'minimum' 'maximum'
Data variables: (12/16)
    time       (index) datetime64[ns] 2013-12-01 2023-12-31
    longitude  (index) float64 -12.0 9.0
    latitude   (index) float64 48.0 64.0
    hs         (index) float64 0.0 13.42
    hswe       (index) float64 0.0 12.44
    tmm10      (index) float64 0.0 50.0
    ...         ...
    ssh        (index) float64 -6.416 7.284
    xcur       (index) float64 -2.999 2.997
    ycur       (index) float64 -2.999 3.0
    xwnd       (index) float64 -28.13 33.3
    ywnd       (index) float64 -31.77 32.19
    wl         (index) float64 -70.36 4.279e+03

In [16]:
ds_min_max_prev = xr.open_dataset(ds_min_max_dir / 'min-max-values.nc')

In [17]:
ds_min_max_all = ds_min_max.copy(deep=True)
for var in ds_min_max.variables:
    ds_min_max_all[var][0] = min(ds_min_max[var][0], ds_min_max_prev[var][0])
    ds_min_max_all[var][1] = max(ds_min_max[var][1], ds_min_max_prev[var][1])

TypeError: IndexVariable values cannot be modified

In [18]:
min_max_values_detailed = {}
for var in ds_min_max.variables:
    min_value = min(ds_min_max[var].values[0], ds_min_max_prev[var].values[0])
    max_value = max(ds_min_max[var].values[1], ds_min_max_prev[var].values[1])
    min_max_values_detailed[var] = {
        "minimum": min_value,
        "maximum": max_value
    }

In [19]:
df_all = pd.DataFrame(min_max_values_detailed)
ds_min_max_all = xr.Dataset.from_dataframe(df_all)
ds_min_max.to_netcdf(ds_min_max_dir / "min-max-values-all.nc")

In [21]:
ds_min_max_all

<xarray.Dataset>
Dimensions:    (index: 2)
Coordinates:
  * index      (index) object 'minimum' 'maximum'
Data variables: (12/16)
    time       (index) datetime64[ns] 2013-12-01 2023-12-31
    longitude  (index) float64 -12.0 9.0
    latitude   (index) float64 48.0 64.0
    hs         (index) float64 0.0 15.95
    hswe       (index) float64 0.0 14.69
    tmm10      (index) float64 0.0 50.0
    ...         ...
    ssh        (index) float64 -6.416 7.284
    xcur       (index) float64 -2.999 3.0
    ycur       (index) float64 -2.999 3.0
    xwnd       (index) float64 -28.13 39.34
    ywnd       (index) float64 -34.12 46.39
    wl         (index) float64 -70.36 4.279e+03

In [22]:
ds_min_max_prev['ywnd'].values

array([-34.11969376,  46.38508224])

In [23]:
ds_min_max

<xarray.Dataset>
Dimensions:    (index: 2)
Coordinates:
  * index      (index) object 'minimum' 'maximum'
Data variables: (12/16)
    time       (index) datetime64[ns] 2013-12-01 2023-12-31
    longitude  (index) float64 -12.0 9.0
    latitude   (index) float64 48.0 64.0
    hs         (index) float64 0.0 13.42
    hswe       (index) float64 0.0 12.44
    tmm10      (index) float64 0.0 50.0
    ...         ...
    ssh        (index) float64 -6.416 7.284
    xcur       (index) float64 -2.999 2.997
    ycur       (index) float64 -2.999 3.0
    xwnd       (index) float64 -28.13 33.3
    ywnd       (index) float64 -31.77 32.19
    wl         (index) float64 -70.36 4.279e+03

In [24]:
min_max_values_detailed = {}
for ens in min_max_values.keys():
    for var in min_max_values[ens].keys():
        try:
            min_max_values_detailed[var] = np.concatenate([min_max_values_detailed[var], min_max_values[ens][var]])
        except Exception as e:
            print(e)
            min_max_values_detailed[var] = min_max_values[ens][var]

'time'
'longitude'
'latitude'
'hs'
'hswe'
'tmm10'
'tps'
'theta0'
'fdir'
'spread'
'ssh'
'xcur'
'ycur'
'xwnd'
'ywnd'
'wl'


In [25]:
min_max_values_detailed

{'time': array(['2023-02-01T00:00:00.000000000', '2023-02-28T00:00:00.000000000',
        '2023-12-01T00:00:00.000000000', '2023-12-31T00:00:00.000000000',
        '2019-01-01T00:00:00.000000000', '2019-02-01T00:00:00.000000000',
        '2017-10-01T00:00:00.000000000', '2017-11-01T00:00:00.000000000',
        '2020-02-01T00:00:00.000000000', '2020-02-29T00:00:00.000000000',
        '2018-01-01T00:00:00.000000000', '2018-02-01T00:00:00.000000000',
        '2013-12-01T00:00:00.000000000', '2013-12-07T00:00:00.000000000'],
       dtype='datetime64[ns]'),
 'longitude': array([-12.,   9., -12.,   9., -12.,   9., -12.,   9., -12.,   9., -12.,
          9., -12.,   9.]),
 'latitude': array([48., 64., 48., 64., 48., 64., 48., 64., 48., 64., 48., 64., 48.,
        64.]),
 'hs': array([ 0.        , 10.68300438,  0.        ,  9.97497463,  0.        ,
        10.13214493,  0.        , 11.009552  ,  0.        , 12.09524822,
         0.        , 13.41517353,  0.        , 11.14841175]),
 'hswe': arr